# MLOps Pipeline: Revenue Forecasting in Snowflake
*Co-authored with CoCo*

End-to-end MLOps lifecycle for daily revenue forecasting using scikit-learn/XGBoost — from EDA through model serving — all within Snowflake.

**Sections:**
1. Exploratory Data Analysis
2. Feature Engineering
3. Model Training & Evaluation
4. Model Registry & Warehouse Inference
5. Batch Scoring (Forecast Generation)
6. Model Monitoring & Drift Detection (automated with Tasks + Alerts)

---
## 0. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

from snowflake.snowpark.context import get_active_session
session = get_active_session()

session.sql("USE DATABASE COCO_DS_DEMO").collect()
session.sql("USE SCHEMA PUBLIC").collect()
session.sql("USE WAREHOUSE COCO_DS_WH").collect()
print(f"Connected | Role: {session.get_current_role()} | DB: {session.get_current_database()}")

---
## 1. Exploratory Data Analysis

Load the DAILY_REVENUE table and explore revenue patterns — trends, seasonality, channel mix, and anomalies across 5 years of daily data.

In [ ]:
# Load daily revenue data
df = session.table("DAILY_REVENUE").to_pandas()
df['DATE'] = pd.to_datetime(df['DATE'])
df = df.sort_values(['DATE', 'CHANNEL']).reset_index(drop=True)

print(f"Shape: {df.shape}")
print(f"Date range: {df['DATE'].min().date()} to {df['DATE'].max().date()}")
print(f"Days: {df['DATE'].nunique()} | Channels: {df['CHANNEL'].unique().tolist()}")
print(f"\nRevenue stats per channel:")
print(df.groupby('CHANNEL')['REVENUE'].describe().round(2).to_string())

In [ ]:
# Revenue over time by channel (line chart)
fig, ax = plt.subplots(figsize=(14, 4))
for ch in sorted(df['CHANNEL'].unique()):
    ch_data = df[df['CHANNEL'] == ch]
    ax.plot(ch_data['DATE'], ch_data['REVENUE'], label=ch, alpha=0.7, linewidth=0.8)
ax.set_title('Daily Revenue by Channel (2020-2024)')
ax.set_xlabel('Date'); ax.set_ylabel('Revenue ($)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Weekly seasonality + monthly trend
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Average revenue by day-of-week
df['DOW'] = df['DATE'].dt.dayofweek
dow_names = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
for ch in sorted(df['CHANNEL'].unique()):
    dow_avg = df[df['CHANNEL']==ch].groupby('DOW')['REVENUE'].mean()
    axes[0].plot(range(7), dow_avg.values, marker='o', label=ch)
axes[0].set_xticks(range(7)); axes[0].set_xticklabels(dow_names)
axes[0].set_title('Weekly Seasonality'); axes[0].set_ylabel('Avg Revenue')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Monthly trend (total)
monthly = df.groupby(df['DATE'].dt.to_period('M'))['REVENUE'].sum()
axes[1].plot(monthly.index.to_timestamp(), monthly.values, color='steelblue')
axes[1].set_title('Monthly Revenue Trend'); axes[1].set_ylabel('Total Revenue')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Distribution per channel + anomaly identification
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.boxplot(data=df, x='CHANNEL', y='REVENUE', ax=axes[0], palette='Set2')
axes[0].set_title('Revenue Distribution by Channel')

# Flag anomalies: values > 3 std from channel mean
anomalies = []
for ch in df['CHANNEL'].unique():
    ch_data = df[df['CHANNEL'] == ch]
    mean, std = ch_data['REVENUE'].mean(), ch_data['REVENUE'].std()
    anom = ch_data[(ch_data['REVENUE'] > mean + 3*std) | (ch_data['REVENUE'] < mean - 3*std)]
    anomalies.append(anom)
anom_df = pd.concat(anomalies)

# Plot total with anomalies marked
total_daily = df.groupby('DATE')['REVENUE'].sum()
axes[1].plot(total_daily.index, total_daily.values, color='steelblue', alpha=0.7, linewidth=0.6)
anom_dates = anom_df.groupby('DATE')['REVENUE'].sum()
for d in anom_dates.index:
    if d in total_daily.index:
        axes[1].scatter(d, total_daily[d], color='red', s=30, zorder=5)
axes[1].set_title(f'Total Revenue with Anomalies ({len(anom_df)} flagged)')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()
print(f"Anomalies: {len(anom_df)} records on {anom_df['DATE'].nunique()} unique dates")

---
## 2. Feature Engineering

Create time-series features for forecasting:
- **Lag features**: yesterday's revenue shifted by 7, 14, 30 days
- **Rolling averages**: smoothed 7-day and 30-day windows
- **Calendar features**: day of week, month, weekend/month boundaries
- **Channel encoding**: one-hot for channel
- **Target**: next-day revenue (shift revenue by -1)

In [ ]:
# Feature engineering per channel (lags/rolling must be channel-specific)
def create_features(group):
    g = group.sort_values('DATE').copy()
    g['REVENUE_LAG_7'] = g['REVENUE'].shift(7)
    g['REVENUE_LAG_14'] = g['REVENUE'].shift(14)
    g['REVENUE_LAG_30'] = g['REVENUE'].shift(30)
    g['ROLLING_7D_AVG'] = g['REVENUE'].shift(1).rolling(7).mean()
    g['ROLLING_30D_AVG'] = g['REVENUE'].shift(1).rolling(30).mean()
    g['TARGET'] = g['REVENUE'].shift(-1)  # next-day revenue
    return g

df_feat = df.groupby('CHANNEL', group_keys=False).apply(create_features)

# Calendar features
df_feat['DAY_OF_WEEK'] = df_feat['DATE'].dt.dayofweek
df_feat['MONTH'] = df_feat['DATE'].dt.month
df_feat['IS_WEEKEND'] = (df_feat['DAY_OF_WEEK'] >= 5).astype(int)
df_feat['IS_MONTH_START'] = df_feat['DATE'].dt.is_month_start.astype(int)
df_feat['IS_MONTH_END'] = df_feat['DATE'].dt.is_month_end.astype(int)

# One-hot encode channel
df_feat = pd.get_dummies(df_feat, columns=['CHANNEL'], dtype=int)

# Drop rows with NaN from lag/rolling + missing target
df_feat = df_feat.dropna(subset=['REVENUE_LAG_30', 'ROLLING_30D_AVG', 'TARGET']).reset_index(drop=True)

# Define feature columns
feature_cols = ['REVENUE_LAG_7', 'REVENUE_LAG_14', 'REVENUE_LAG_30',
                'ROLLING_7D_AVG', 'ROLLING_30D_AVG',
                'DAY_OF_WEEK', 'MONTH', 'IS_WEEKEND', 'IS_MONTH_START', 'IS_MONTH_END',
                'CHANNEL_atm', 'CHANNEL_in_store', 'CHANNEL_online']

print(f"Feature-engineered dataset: {df_feat.shape}")
print(f"Date range: {df_feat['DATE'].min().date()} to {df_feat['DATE'].max().date()}")
print(f"\nFeatures ({len(feature_cols)}): {feature_cols}")
print(f"Target: next-day revenue")

---
## 3. Model Training & Evaluation

Temporal train/test split — hold out the **last 30 days** as test set (no shuffling, preserving time order).

Compare three regressors:
- **XGBRegressor** — gradient-boosted trees
- **RandomForestRegressor** — ensemble of decision trees
- **Ridge** — regularized linear baseline

Metrics: MAE, RMSE, MAPE, R²

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor

# Temporal split: last 30 days as test
cutoff_date = df_feat['DATE'].max() - timedelta(days=30)
train = df_feat[df_feat['DATE'] <= cutoff_date]
test = df_feat[df_feat['DATE'] > cutoff_date]

X_train, y_train = train[feature_cols], train['TARGET']
X_test, y_test = test[feature_cols], test['TARGET']

print(f"Train: {len(X_train):,} rows ({train['DATE'].min().date()} to {train['DATE'].max().date()})")
print(f"Test:  {len(X_test):,} rows ({test['DATE'].min().date()} to {test['DATE'].max().date()})")

In [ ]:
def mape(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

# Train three models
models = {
    'XGBoost': XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.05,
                            subsample=0.8, random_state=42, verbosity=0),
    'Random Forest': RandomForestRegressor(n_estimators=200, max_depth=10,
                                           random_state=42, n_jobs=-1),
    'Ridge': Ridge(alpha=1.0)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    results[name] = {
        'model': model,
        'mae': mean_absolute_error(y_test, y_pred),
        'rmse': np.sqrt(mean_squared_error(y_test, y_pred)),
        'mape': mape(y_test.values, y_pred),
        'r2': r2_score(y_test, y_pred),
        'y_pred': y_pred
    }

metrics_df = pd.DataFrame({k: {m: v for m, v in r.items() if m not in ['model','y_pred']}
                           for k, r in results.items()}).T
print(metrics_df.round(4).to_string())

best_name = metrics_df['mape'].idxmin()
best_model = results[best_name]['model']
print(f"\n>>> Best model: {best_name} (MAPE={results[best_name]['mape']:.2f}%, R²={results[best_name]['r2']:.4f})")

In [ ]:
# Predicted vs Actual for each model
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
colors = {'XGBoost': 'coral', 'Random Forest': 'steelblue', 'Ridge': 'green'}

test_dates = test['DATE'].values
for idx, (name, r) in enumerate(results.items()):
    axes[idx].plot(test_dates, y_test.values, 'k-', alpha=0.7, label='Actual', linewidth=1)
    axes[idx].plot(test_dates, r['y_pred'], color=colors[name], alpha=0.8, label=f'{name} (MAPE={r["mape"]:.1f}%)')
    axes[idx].set_title(name); axes[idx].legend(loc='upper right'); axes[idx].grid(alpha=0.3)

axes[2].set_xlabel('Date')
plt.suptitle('Predicted vs Actual Revenue (Test Period)', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance (top 10) for best model
if hasattr(best_model, 'feature_importances_'):
    imp = pd.Series(best_model.feature_importances_, index=feature_cols)
else:
    imp = pd.Series(np.abs(best_model.coef_), index=feature_cols)

top10 = imp.sort_values(ascending=False).head(10)
fig, ax = plt.subplots(figsize=(8, 4))
top10.sort_values().plot(kind='barh', ax=ax, color='steelblue')
ax.set_title(f'Top 10 Feature Importances ({best_name})')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

---
## 4. Model Registry & Warehouse Inference

Register the best model in Snowflake's Model Registry and demonstrate warehouse-based inference.

**Key points:**
- `target_platforms=["WAREHOUSE"]` — runs on the warehouse, no container service needed
- `mv.run(data, function_name="predict")` — direct warehouse inference
- `create_service()` is for SPCS container deployments only

In [ ]:
from snowflake.ml.registry import Registry

# Initialize registry
reg = Registry(session=session, database_name="COCO_DS_DEMO", schema_name="PUBLIC")

# Prepare metrics and sample
training_metrics = {
    'mae': float(results[best_name]['mae']),
    'rmse': float(results[best_name]['rmse']),
    'mape': float(results[best_name]['mape']),
    'r2': float(results[best_name]['r2'])
}
sample_input = X_test.head(10)

# Log model
mv = reg.log_model(
    model=best_model,
    model_name="revenue_forecaster",
    version_name="v1",
    metrics=training_metrics,
    sample_input_data=sample_input,
    target_platforms=["WAREHOUSE"]
)
print(f"Model registered: revenue_forecaster v1")
print(f"Metrics: {training_metrics}")

In [ ]:
# List registered models and versions
print("Registered models:")
print(reg.show_models()[['name', 'default_version_name']].to_string())

model_ref = reg.get_model("revenue_forecaster")
print(f"\nVersions for revenue_forecaster:")
print(model_ref.show_versions()[['name', 'is_default_version']].to_string())

In [ ]:
# Warehouse-based inference
sample = X_test.head(5)
predictions = mv.run(sample, function_name="predict")
print("Warehouse inference (predict):")
print(predictions)

print("\n--- SQL equivalent ---")
print("SELECT COCO_DS_DEMO.PUBLIC.REVENUE_FORECASTER!PREDICT(...)")
print("-- Pass feature columns as function arguments")
print("\nNote: create_service() is for SPCS container deployments only.")
print("Warehouse inference uses mv.run() or MODEL!FUNCTION() SQL syntax.")

---
## 5. Batch Scoring (Forecast Generation)

Generate a 30-day revenue forecast, write to Snowflake, and compare against held-out actuals.

In [ ]:
# Score the test period (next 30 days) using the best model
forecast_pred = best_model.predict(X_test)

# Build output dataframe
forecast_df = test[['DATE']].copy()
# Recover channel from one-hot columns
forecast_df['CHANNEL'] = test[['CHANNEL_atm','CHANNEL_in_store','CHANNEL_online']].idxmax(axis=1).str.replace('CHANNEL_', '')
forecast_df['PREDICTED_REVENUE'] = np.round(forecast_pred, 2)
forecast_df['SCORED_AT'] = datetime.now()

# Write to Snowflake
session.create_dataframe(forecast_df).write.mode("overwrite").save_as_table(
    "COCO_DS_DEMO.PUBLIC.REVENUE_FORECAST"
)
print(f"Forecast written: {len(forecast_df)} rows to REVENUE_FORECAST")
print(forecast_df.head(10).to_string(index=False))

In [ ]:
# Compare forecast vs actuals: MAPE per channel
forecast_df['ACTUAL'] = y_test.values
forecast_df['ERROR_PCT'] = np.abs((forecast_df['ACTUAL'] - forecast_df['PREDICTED_REVENUE']) / forecast_df['ACTUAL']) * 100

print("Forecast Accuracy by Channel:")
channel_mape = forecast_df.groupby('CHANNEL')['ERROR_PCT'].mean()
for ch, m in channel_mape.items():
    print(f"  {ch:>10}: MAPE = {m:.2f}%")
print(f"  {'Overall':>10}: MAPE = {forecast_df['ERROR_PCT'].mean():.2f}%")

# Plot forecast vs actual with confidence band
fig, ax = plt.subplots(figsize=(14, 5))
for ch in sorted(forecast_df['CHANNEL'].unique()):
    ch_data = forecast_df[forecast_df['CHANNEL'] == ch].sort_values('DATE')
    ax.plot(ch_data['DATE'], ch_data['ACTUAL'], 'k-', alpha=0.5, linewidth=0.8)
    ax.plot(ch_data['DATE'], ch_data['PREDICTED_REVENUE'], '--', label=f'{ch} forecast', linewidth=1.5)
ax.set_title('Forecast vs Actual (30-day Test Period)')
ax.set_xlabel('Date'); ax.set_ylabel('Revenue ($)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Plot full history + forecast
fig, ax = plt.subplots(figsize=(14, 5))

# Historical (all training data, total revenue per day)
hist = df_feat[df_feat['DATE'] <= cutoff_date].groupby('DATE')['REVENUE'].sum()
ax.plot(hist.index, hist.values, color='steelblue', alpha=0.5, linewidth=0.6, label='Historical')

# Forecast period
fcast = forecast_df.groupby('DATE').agg({'PREDICTED_REVENUE':'sum', 'ACTUAL':'sum'}).reset_index()
ax.plot(fcast['DATE'], fcast['ACTUAL'], 'k-', linewidth=1.5, label='Actual (test)')
ax.plot(fcast['DATE'], fcast['PREDICTED_REVENUE'], 'r--', linewidth=1.5, label='Forecast')
ax.fill_between(fcast['DATE'], fcast['PREDICTED_REVENUE']*0.9, fcast['PREDICTED_REVENUE']*1.1,
                alpha=0.15, color='red', label='±10% band')

ax.axvline(cutoff_date, color='gray', linestyle=':', label='Forecast start')
ax.set_title('Full History + 30-Day Forecast')
ax.legend(loc='upper left'); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## 6. Model Monitoring & Drift Detection

Simulate production monitoring:
1. Generate a "next week" of revenue with shifted distributions
2. Score and measure forecast accuracy degradation
3. Detect **data drift** (feature distribution shifts via KS test)
4. Detect **prediction drift** (forecast distribution shift)
5. Log to monitoring table
6. Create automated **Snowflake Task** and **Alert**

In [ ]:
from scipy import stats

# Simulate next week of actuals with drift:
# - Revenue shifted 20% higher (sudden growth)
# - More variance (unstable pattern)
np.random.seed(42)
n_days = 7
channels = ['atm', 'in_store', 'online']
shares = {'atm': 0.15, 'in_store': 0.35, 'online': 0.50}

# Use last known features as base and shift revenue up
last_features = X_test.tail(3 * n_days).copy()  # last week of test for each channel

# Simulate drifted actuals (shifted 20% up with extra noise)
simulated_actuals = y_test.tail(3 * n_days).values * (1.2 + np.random.normal(0, 0.1, 3 * n_days))

# Score with our model
new_predictions = best_model.predict(last_features)

print(f"Simulated week: {3*n_days} records")
print(f"Actual mean: ${simulated_actuals.mean():,.0f} (shifted +20% from training)")
print(f"Predicted mean: ${new_predictions.mean():,.0f}")
print(f"MAPE on new week: {mape(simulated_actuals, new_predictions):.2f}%")

In [ ]:
# PERFORMANCE MONITORING: MAPE comparison
new_mape = mape(simulated_actuals, new_predictions)
mape_threshold = training_metrics['mape'] * 1.5  # 50% degradation threshold

print("PERFORMANCE MONITORING:")
print(f"{'Metric':<18} {'Training':>10} {'Current':>10} {'Threshold':>10} {'Alert':>6}")
print("-" * 58)
alert_perf = new_mape > mape_threshold
print(f"{'MAPE (%)':<18} {training_metrics['mape']:>10.2f} {new_mape:>10.2f} {mape_threshold:>10.2f} {'⚠️' if alert_perf else '':>6}")

new_r2 = r2_score(simulated_actuals, new_predictions)
print(f"{'R²':<18} {training_metrics['r2']:>10.4f} {new_r2:>10.4f} {'':>10} {'':>6}")

In [ ]:
# DATA DRIFT: KS test on key features
drift_features = ['REVENUE_LAG_7', 'REVENUE_LAG_14', 'ROLLING_7D_AVG', 'ROLLING_30D_AVG', 'DAY_OF_WEEK']

drift_results = []
for feat in drift_features:
    ks_stat, ks_pval = stats.ks_2samp(X_train[feat].values, last_features[feat].values)
    drift_results.append({'feature': feat, 'ks_stat': ks_stat, 'p_value': ks_pval,
                          'drift_detected': ks_pval < 0.05})

drift_df = pd.DataFrame(drift_results)
print("DATA DRIFT (KS Test):")
print(drift_df.to_string(index=False))

# Visualize drift
fig, ax = plt.subplots(figsize=(8, 4))
colors = ['coral' if d else 'steelblue' for d in drift_df['drift_detected']]
ax.barh(drift_df['feature'], drift_df['ks_stat'], color=colors)
ax.axvline(0.1, color='red', linestyle='--', label='Threshold')
ax.set_xlabel('KS Statistic')
ax.set_title('Data Drift Detection (KS Test)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# PREDICTION DRIFT: compare training predictions vs new predictions
train_preds = best_model.predict(X_train)
pred_ks, pred_pval = stats.ks_2samp(train_preds, new_predictions)

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(train_preds, bins=50, alpha=0.6, label='Training predictions', density=True, color='steelblue')
ax.hist(new_predictions, bins=20, alpha=0.6, label='New period predictions', density=True, color='coral')
ax.set_title(f'Prediction Drift (KS={pred_ks:.3f}, p={pred_pval:.2e})')
ax.set_xlabel('Predicted Revenue'); ax.set_ylabel('Density')
ax.legend()
plt.tight_layout()
plt.show()
print(f"Prediction drift: KS={pred_ks:.4f}, p={pred_pval:.2e}, Drift={'YES' if pred_pval < 0.05 else 'No'}")

In [ ]:
# Save monitoring results to Snowflake
monitoring_records = []
run_date = datetime.now()

# Performance
for ch in channels:
    monitoring_records.append({
        'RUN_DATE': run_date, 'CHANNEL': ch,
        'METRIC_NAME': 'mape', 'TRAINING_VALUE': float(training_metrics['mape']),
        'CURRENT_VALUE': float(new_mape), 'DRIFT_SCORE': float(abs(new_mape - training_metrics['mape'])),
        'ALERT_FLAG': bool(alert_perf)
    })

# Data drift
for _, row in drift_df.iterrows():
    monitoring_records.append({
        'RUN_DATE': run_date, 'CHANNEL': 'all',
        'METRIC_NAME': f'drift_{row["feature"]}', 'TRAINING_VALUE': 0.0,
        'CURRENT_VALUE': float(row['ks_stat']), 'DRIFT_SCORE': float(row['ks_stat']),
        'ALERT_FLAG': bool(row['drift_detected'])
    })

# Prediction drift
monitoring_records.append({
    'RUN_DATE': run_date, 'CHANNEL': 'all',
    'METRIC_NAME': 'prediction_drift_ks', 'TRAINING_VALUE': 0.0,
    'CURRENT_VALUE': float(pred_ks), 'DRIFT_SCORE': float(pred_ks),
    'ALERT_FLAG': pred_pval < 0.05
})

mon_df = pd.DataFrame(monitoring_records)
session.create_dataframe(mon_df).write.mode("overwrite").save_as_table(
    "COCO_DS_DEMO.PUBLIC.FORECAST_MONITORING_LOG"
)
print(f"Monitoring log: {len(mon_df)} records written")
print(f"Alerts triggered: {mon_df['ALERT_FLAG'].sum()}")
print(mon_df[['CHANNEL','METRIC_NAME','TRAINING_VALUE','CURRENT_VALUE','DRIFT_SCORE','ALERT_FLAG']].to_string(index=False))

### Automated Monitoring: Snowflake Task & Alert

Create infrastructure for automated daily monitoring:
- **Stored Procedure** — computes forecast accuracy and drift on new data
- **Task** — runs the procedure daily at 7AM UTC
- **Alert** — fires when MAPE exceeds threshold or drift is detected

In [ ]:
%%sql -r create_mon_table
CREATE OR REPLACE TABLE COCO_DS_DEMO.PUBLIC.FORECAST_MONITORING_LOG (
    RUN_DATE TIMESTAMP_NTZ,
    CHANNEL VARCHAR,
    METRIC_NAME VARCHAR,
    TRAINING_VALUE FLOAT,
    CURRENT_VALUE FLOAT,
    DRIFT_SCORE FLOAT,
    ALERT_FLAG BOOLEAN
)

In [ ]:
%%sql -r create_proc
CREATE OR REPLACE PROCEDURE COCO_DS_DEMO.PUBLIC.RUN_FORECAST_MONITORING()
RETURNS VARCHAR
LANGUAGE SQL
EXECUTE AS CALLER
AS
BEGIN
    LET v_run_date TIMESTAMP := CURRENT_TIMESTAMP();
    LET v_baseline_mape FLOAT := 8.0;
    LET v_mape_threshold FLOAT := 12.0;
    
    -- Compare forecast vs actuals for recent period
    LET v_current_mape FLOAT := (
        SELECT AVG(ABS(f.PREDICTED_REVENUE - r.REVENUE) / NULLIF(r.REVENUE, 0)) * 100
        FROM COCO_DS_DEMO.PUBLIC.REVENUE_FORECAST f
        JOIN COCO_DS_DEMO.PUBLIC.DAILY_REVENUE r
            ON f.DATE = r.DATE AND f.CHANNEL = r.CHANNEL
    );
    
    LET v_mape_drift FLOAT := ABS(:v_current_mape - :v_baseline_mape);
    LET v_mape_alert BOOLEAN := (:v_current_mape > :v_mape_threshold);
    
    -- Get revenue distribution drift
    LET v_avg_predicted FLOAT := (
        SELECT AVG(PREDICTED_REVENUE) FROM COCO_DS_DEMO.PUBLIC.REVENUE_FORECAST
    );
    LET v_avg_actual FLOAT := (
        SELECT AVG(REVENUE) FROM COCO_DS_DEMO.PUBLIC.DAILY_REVENUE
        WHERE DATE >= DATEADD('day', -30, CURRENT_DATE())
    );
    LET v_rev_drift FLOAT := ABS(:v_avg_predicted - :v_avg_actual) / NULLIF(:v_avg_actual, 0);
    LET v_rev_alert BOOLEAN := (:v_rev_drift > 0.15);
    
    INSERT INTO COCO_DS_DEMO.PUBLIC.FORECAST_MONITORING_LOG
        (RUN_DATE, CHANNEL, METRIC_NAME, TRAINING_VALUE, CURRENT_VALUE, DRIFT_SCORE, ALERT_FLAG)
    VALUES (:v_run_date, 'all', 'mape', :v_baseline_mape, :v_current_mape, :v_mape_drift, :v_mape_alert);
    
    INSERT INTO COCO_DS_DEMO.PUBLIC.FORECAST_MONITORING_LOG
        (RUN_DATE, CHANNEL, METRIC_NAME, TRAINING_VALUE, CURRENT_VALUE, DRIFT_SCORE, ALERT_FLAG)
    VALUES (:v_run_date, 'all', 'revenue_drift', :v_avg_actual, :v_avg_predicted, :v_rev_drift, :v_rev_alert);
    
    RETURN 'Forecast monitoring complete. MAPE=' || :v_current_mape::VARCHAR || '%, Alert=' || :v_mape_alert::VARCHAR;
END;

In [ ]:
%%sql -r create_task
-- Daily monitoring task
CREATE OR REPLACE TASK COCO_DS_DEMO.PUBLIC.DAILY_FORECAST_MONITORING_TASK
    WAREHOUSE = COCO_DS_WH
    SCHEDULE = 'USING CRON 0 7 * * * UTC'
    COMMENT = 'Daily revenue forecast monitoring'
AS
    CALL COCO_DS_DEMO.PUBLIC.RUN_FORECAST_MONITORING();

ALTER TASK COCO_DS_DEMO.PUBLIC.DAILY_FORECAST_MONITORING_TASK RESUME

In [ ]:
%%sql -r create_alert
-- Alert when MAPE exceeds threshold or drift detected
CREATE OR REPLACE ALERT COCO_DS_DEMO.PUBLIC.FORECAST_DRIFT_ALERT
    WAREHOUSE = COCO_DS_WH
    SCHEDULE = 'USING CRON 30 7 * * * UTC'
    IF (EXISTS (
        SELECT 1 FROM COCO_DS_DEMO.PUBLIC.FORECAST_MONITORING_LOG
        WHERE RUN_DATE > DATEADD('hour', -2, CURRENT_TIMESTAMP())
          AND ALERT_FLAG = TRUE
    ))
    THEN
        CALL SYSTEM$SEND_EMAIL(
            'forecast_monitoring_integration',
            'data-team@company.com',
            'ALERT: Revenue Forecast Drift Detected',
            'MAPE exceeded threshold or distribution drift detected. Check FORECAST_MONITORING_LOG.'
        )

In [ ]:
%%sql -r test_monitoring
-- Test the procedure
CALL COCO_DS_DEMO.PUBLIC.RUN_FORECAST_MONITORING();

SELECT * FROM COCO_DS_DEMO.PUBLIC.FORECAST_MONITORING_LOG
ORDER BY RUN_DATE DESC LIMIT 5

In [ ]:
# Final summary
print("=" * 60)
print("  Revenue Forecasting Pipeline Summary")
print("=" * 60)
print(f"\n{'Dataset:':<28} COCO_DS_DEMO.PUBLIC.DAILY_REVENUE")
print(f"{'Rows:':<28} {len(df):,} (5 years, 3 channels)")
print(f"{'Date range:':<28} 2020-01-01 to 2024-12-30")
print(f"\n{'Best Model:':<28} {best_name}")
print(f"{'MAPE:':<28} {training_metrics['mape']:.2f}%")
print(f"{'R²:':<28} {training_metrics['r2']:.4f}")
print(f"\n{'Model Registry:':<28} revenue_forecaster v1")
print(f"{'Forecast Table:':<28} REVENUE_FORECAST")
print(f"{'Monitoring Log:':<28} FORECAST_MONITORING_LOG")
print(f"\n{'Monitoring Task:':<28} DAILY_FORECAST_MONITORING_TASK (7:00 AM UTC)")
print(f"{'Drift Alert:':<28} FORECAST_DRIFT_ALERT (7:30 AM UTC)")
print(f"{'Drift features detected:':<28} {drift_df['drift_detected'].sum()} / {len(drift_df)}")
print("\n" + "=" * 60)
print("  Pipeline Complete!")
print("=" * 60)